<a href="https://colab.research.google.com/github/TesterSim2/Focused-IDE/blob/main/B2B_Business_Intent_Mapping_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# CELL 1: DEPENDENCIES & ENVIRONMENT INITIALIZATION
# ==============================================================================

# 1. Install required production dependencies
# - tree-sitter & tree-sitter-python: For deterministic, language-agnostic AST parsing
# - networkx: For constructing the mathematical Directed Graph
# - vllm: High-throughput memory-efficient LLM inference engine (perfect for A100)
# - pydantic: For strict, production-grade schema validation
# - loguru: For concurrent, thread-safe production logging
!pip install -q tree-sitter==0.21.3 tree-sitter-python==0.21.0 networkx vllm transformers pydantic loguru nest_asyncio

import os
import json
import asyncio
import nest_asyncio
import networkx as nx
from typing import List, Dict, Optional, Any, Set
from pydantic import BaseModel, Field

# Tree-sitter for deterministic parsing
import tree_sitter
import tree_sitter_python

# Production logging
from loguru import logger
import sys

# Patch asyncio to allow nested event loops in Jupyter/Colab environments
# (Required for vLLM asynchronous engines in Colab)
nest_asyncio.apply()

# Configure production logger
logger.remove()
logger.add(
    sys.stdout,
    format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - <level>{message}</level>",
    level="INFO"
)

# Initialize the deterministic parser for Python
try:
    LANGUAGE_PYTHON = tree_sitter.Language(tree_sitter_python.language(), "python")
    parser = tree_sitter.Parser()
    parser.set_language(LANGUAGE_PYTHON)
    logger.info("Tree-sitter deterministic parser successfully initialized for Python.")
except Exception as e:
    logger.error(f"Failed to initialize Tree-sitter: {e}")

# Hardware Verification
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    logger.info(f"Hardware Verified: Detected {gpu_name} with {vram_gb:.2f} GB VRAM.")
    if vram_gb < 79.0:
        logger.warning("VRAM is less than 80GB. We may need to use quantization for the 32B model.")
else:
    logger.critical("CRITICAL: No GPU detected. Please ensure your Colab runtime is set to A100.")

logger.info("Cell 1 Execution Complete. Environment is primed for Neuro-Symbolic Graph generation.")

2026-05-07 22:24:30 | INFO     | __main__:<cell line: 0>:46 - Tree-sitter deterministic parser successfully initialized for Python.
2026-05-07 22:24:31 | INFO     | __main__:<cell line: 0>:55 - Hardware Verified: Detected NVIDIA A100-SXM4-80GB with 79.25 GB VRAM.
2026-05-07 22:24:31 | INFO     | __main__:<cell line: 0>:61 - Cell 1 Execution Complete. Environment is primed for Neuro-Symbolic Graph generation.


In [2]:
# ==============================================================================
# CELL 2: STRICT SCHEMAS & TARGET CODEBASE (THE "VIBE CODE" ATTACK)
# ==============================================================================

from enum import Enum
from pydantic import BaseModel, Field
from typing import List, Optional

# 1. The Paper 2 Logical Error Taxonomy
# We strictly define the 10 errors. We will anonymize these in the actual prompt,
# but we enforce the schema here so the LLM output can be programmatically verified.
class LogicalErrorType(str, Enum):
    ERROR_A_INPUT = "Error_A_Input"
    ERROR_B_OUTPUT = "Error_B_Output"
    ERROR_C_VARIABLE = "Error_C_Variable"
    ERROR_D_COMPUTATION = "Error_D_Computation"
    ERROR_E_CONDITION = "Error_E_Condition"
    ERROR_F_BRANCHING = "Error_F_Branching"
    ERROR_G_LOOP = "Error_G_Loop"
    ERROR_H_ARRAY_STRING = "Error_H_Array_String"
    ERROR_I_FUNCTION = "Error_I_Function"
    ERROR_J_CONCEPTUAL = "Error_J_Conceptual"
    NO_ERROR = "No_Error"

# 2. The Semantic Graph Node Schema
class SemanticNode(BaseModel):
    node_id: str = Field(..., description="Unique identifier for the function/class")
    node_type: str = Field(..., description="Type of node (e.g., function_definition)")
    raw_code: str = Field(..., description="The raw source code of this node")
    semantic_intent: Optional[str] = Field(None, description="LLM-generated business intent of this node")

# 3. The LLM Structured Output Schema (The Audit Result)
class AuditResult(BaseModel):
    status: str = Field(..., description="Either 'APPROVED' or 'REJECTED'")
    error_type: LogicalErrorType = Field(..., description="The specific logical error detected, or NO_ERROR")
    reasoning: str = Field(..., description="Tree-of-Thoughts reasoning explaining the violation")
    blast_radius_nodes: List[str] = Field(..., description="List of node_ids involved in the violation")

# ==============================================================================
# 4. THE DUMMY CODEBASE (Baseline vs. AI-Generated PR)
# ==============================================================================

# Baseline Code: The architecture intent is that tax is calculated on the BASE price.
BASELINE_CODE = """
def calculate_tax(base_price: float) -> float:
    return base_price * 0.08

def apply_discount(base_price: float, discount: float) -> float:
    return base_price - discount

def process_order(base_price: float, discount: float) -> float:
    tax = calculate_tax(base_price)
    discounted_price = apply_discount(base_price, discount)
    return discounted_price + tax
"""

# The AI-Generated PR (Vibe Code)
# FLAW: The AI altered the data flow. It now applies the discount first, and
# calculates the tax on the DISCOUNTED price. This compiles perfectly, unit tests
# checking for return type 'float' will pass, but it breaks the domain business logic
# (tax fraud/undercharging). This is a Type D (Computation) / Type J (Conceptual) error.
PR_CODE = """
def calculate_tax(base_price: float) -> float:
    return base_price * 0.08

def apply_discount(base_price: float, discount: float) -> float:
    return base_price - discount

def process_order(base_price: float, discount: float) -> float:
    discounted_price = apply_discount(base_price, discount)
    tax = calculate_tax(discounted_price)
    return discounted_price + tax
"""

logger.info("Cell 2 Execution Complete. Pydantic schemas and dummy codebase successfully initialized.")

2026-05-07 22:24:55 | INFO     | __main__:<cell line: 0>:75 - Cell 2 Execution Complete. Pydantic schemas and dummy codebase successfully initialized.


In [3]:
# ==============================================================================
# CELL 3: ENGINE 1 - DETERMINISTIC AST TO DIRECTED GRAPH PARSER
# ==============================================================================

def build_code_graph(source_code: str, graph_name: str = "CodeGraph") -> nx.DiGraph:
    """
    Parses source code into a Deterministic Directed Graph.
    Nodes = Functions (SemanticNode)
    Edges = Function Calls (Execution / Data Flow)
    """
    G = nx.DiGraph(name=graph_name)

    try:
        # 1. Parse the byte-code into an AST
        tree = parser.parse(bytes(source_code, "utf8"))
        root_node = tree.root_node

        known_functions = {}

        # 2. First Pass: Find all Function Definitions and add them as Nodes
        def extract_nodes(node):
            if node.type == 'function_definition':
                name_node = node.child_by_field_name('name')
                if name_node:
                    func_name = name_node.text.decode('utf8')
                    raw_code = node.text.decode('utf8')

                    # Enforce our strict schema
                    sem_node = SemanticNode(
                        node_id=func_name,
                        node_type='function_definition',
                        raw_code=raw_code
                    )
                    known_functions[func_name] = sem_node
                    G.add_node(func_name, data=sem_node)

            for child in node.children:
                extract_nodes(child)

        extract_nodes(root_node)

        # 3. Second Pass: Find Function Calls to establish Directed Edges
        def extract_edges(node, current_func_name):
            if node.type == 'call':
                func_node = node.child_by_field_name('function')
                # We only map internal function calls to build our blast radius
                if func_node and func_node.type == 'identifier':
                    called_func = func_node.text.decode('utf8')
                    if called_func in known_functions:
                        G.add_edge(current_func_name, called_func)

            for child in node.children:
                extract_edges(child, current_func_name)

        def walk_for_edges(node):
            if node.type == 'function_definition':
                name_node = node.child_by_field_name('name')
                if name_node:
                    current_func = name_node.text.decode('utf8')
                    body_node = node.child_by_field_name('body')
                    if body_node:
                        extract_edges(body_node, current_func)
            for child in node.children:
                walk_for_edges(child)

        walk_for_edges(root_node)

        logger.info(f"Successfully generated {graph_name} | Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")
        return G

    except Exception as e:
        logger.error(f"Failed to parse graph {graph_name}: {e}")
        raise

# Generate the graphs for our Baseline and our PR
baseline_graph = build_code_graph(BASELINE_CODE, graph_name="Baseline_Architecture")
pr_graph = build_code_graph(PR_CODE, graph_name="Pull_Request_Architecture")

# Quick terminal verification of the extracted topology
print("\n--- BASELINE GRAPH TOPOLOGY ---")
for node in baseline_graph.nodes():
    edges = list(baseline_graph.successors(node))
    print(f"Node[{node}] calls -> {edges if edges else 'Nothing (Leaf Node)'}")

print("\n--- PR GRAPH TOPOLOGY ---")
for node in pr_graph.nodes():
    edges = list(pr_graph.successors(node))
    print(f"Node [{node}] calls -> {edges if edges else 'Nothing (Leaf Node)'}")

print("\n")
logger.info("Cell 3 Execution Complete. Deterministic graphs constructed successfully.")

2026-05-07 22:27:27 | INFO     | __main__:build_code_graph:68 - Successfully generated Baseline_Architecture | Nodes: 3 | Edges: 2
2026-05-07 22:27:27 | INFO     | __main__:build_code_graph:68 - Successfully generated Pull_Request_Architecture | Nodes: 3 | Edges: 2

--- BASELINE GRAPH TOPOLOGY ---
Node[calculate_tax] calls -> Nothing (Leaf Node)
Node[apply_discount] calls -> Nothing (Leaf Node)
Node[process_order] calls -> ['calculate_tax', 'apply_discount']

--- PR GRAPH TOPOLOGY ---
Node [calculate_tax] calls -> Nothing (Leaf Node)
Node [apply_discount] calls -> Nothing (Leaf Node)
Node [process_order] calls -> ['apply_discount', 'calculate_tax']


2026-05-07 22:27:27 | INFO     | __main__:<cell line: 0>:91 - Cell 3 Execution Complete. Deterministic graphs constructed successfully.


In [4]:
# ==============================================================================
# CELL 4: ENGINE 2 - LLM INITIALIZATION & SEMANTIC ANNOTATION
# ==============================================================================

import json
import re
from vllm import LLM, SamplingParams

# 1. Initialize the LLM
# We use Qwen2.5-Coder-32B-Instruct. It is the current open-weight king of coding and reasoning.
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

logger.info(f"Loading {MODEL_NAME} into VRAM via vLLM. This will take ~5 minutes to download the weights...")

# Configure vLLM for the A100
llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.90, # Leave 10% VRAM for context window and graph operations
    max_model_len=4096,          # Restrict context window for speed in this experiment
    trust_remote_code=True,
    enforce_eager=True           # Optimizes small-batch latency
)

# We use greedy decoding (temperature=0.0) because we are engineering a deterministic
# audit system, not a creative writing chatbot.
sampling_params = SamplingParams(temperature=0.0, max_tokens=256)

def extract_json(response_text: str) -> dict:
    """Robust JSON extraction for LLM outputs."""
    try:
        match = re.search(r'```json\n(.*?)\n```', response_text, re.DOTALL)
        if match:
            return json.loads(match.group(1))
        return json.loads(response_text)
    except:
        return {"intent": "Failed to parse semantic intent."}

def generate_semantic_intent(graph: nx.DiGraph):
    """
    Traverses the mathematical graph and uses the LLM to attach
    business/semantic intent to every node.
    """
    logger.info("Starting Engine 2: Semantic Annotation Pipeline...")

    prompts =[]
    node_keys = list(graph.nodes())

    for node in node_keys:
        node_data = graph.nodes[node]['data']

        # Strict Prompt Engineering: We force the LLM to act as a tagger, not a coder.
        sys_prompt = "You are a Principal Software Architect. Analyze the following function and define its core business intent in 1 sentence. You must respond ONLY with a valid JSON object matching this schema: {\"intent\": \"string\"}"
        user_prompt = f"Function Code:\n{node_data.raw_code}"

        # Qwen-specific prompt formatting
        prompt = f"<|im_start|>system\n{sys_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n```json\n"
        prompts.append(prompt)

    # 2. Batched Inference
    # We send all nodes to the GPU simultaneously for massive throughput.
    logger.info(f"Executing batched inference for {len(prompts)} nodes...")
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)

    print("\n--- BASELINE SEMANTIC GRAPH ---")
    for idx, output in enumerate(outputs):
        node_name = node_keys[idx]

        # The prompt injected the opening json ticks, so we append them back to parse
        generated_text = output.outputs[0].text
        parsed = extract_json("```json\n" + generated_text)
        intent_str = parsed.get("intent", "No intent extracted.")

        # 3. Inject the soul into the skeleton
        # We update our Pydantic model inside the NetworkX graph
        graph.nodes[node_name]['data'].semantic_intent = intent_str
        print(f"Node [{node_name}] Intent: {intent_str}")

# Run Engine 2 on our Baseline Graph
generate_semantic_intent(baseline_graph)
print("\n")
logger.info("Cell 4 Execution Complete. The Baseline Architecture is now semantically mapped.")

2026-05-07 22:29:09 | INFO     | __main__:<cell line: 0>:13 - Loading Qwen/Qwen2.5-Coder-32B-Instruct into VRAM via vLLM. This will take ~5 minutes to download the weights...
INFO 05-07 22:29:09 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-Coder-32B-Instruct'}


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

INFO 05-07 22:29:27 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-07 22:29:27 [nixl_utils.py:34] NIXL is not available
WARNING 05-07 22:29:27 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-07 22:29:27 [model.py:555] Resolved architecture: Qwen2ForCausalLM
INFO 05-07 22:29:27 [model.py:1680] Using max model len 4096
INFO 05-07 22:29:27 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-07 22:29:27 [vllm.py:840] Asynchronous scheduling is enabled.
WARNING 05-07 22:29:27 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-07 22:29:27 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-07 22:29:27 [kernel.py:205] Final IR op priority after setting platf

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

WARNING 05-07 22:29:31 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
2026-05-07 22:33:17 | INFO     | __main__:generate_semantic_intent:44 - Starting Engine 2: Semantic Annotation Pipeline...
2026-05-07 22:33:17 | INFO     | __main__:generate_semantic_intent:62 - Executing batched inference for 3 nodes...

--- BASELINE SEMANTIC GRAPH ---
Node [calculate_tax] Intent: Calculate the tax amount based on a given base price with a tax rate of 8%.
Node [apply_discount] Intent: Calculate the final price after applying a discount to the base price.
Node [process_order] Intent: Calculate the final price of an order after applying a discount and adding tax.


2026-05-07 22:33:19 | INFO     | __main__:<cell line: 0>:82 - Cell 4 Execution Complete. The Baseline Architecture is now sem

In [9]:
# ==============================================================================
# CELL 5: ENGINE 3 & 4 - BLAST RADIUS & ANONYMIZED LOGICAL AUDITOR (HARDENED)
# ==============================================================================

import json

logger.info("Starting Engine 3: Extracting Graph Delta (Blast Radius)...")

# 1. Engine 3: Extract the Blast Radius
target_node = "process_order"
original_intent = baseline_graph.nodes[target_node]['data'].semantic_intent
new_code = pr_graph.nodes[target_node]['data'].raw_code

# The blast radius includes the mutated node and its immediate execution dependencies
blast_radius = [target_node] + list(pr_graph.successors(target_node))
logger.info(f"Blast Radius identified: {blast_radius}")

# ==============================================================================
# 2. Engine 4: The Paper 2 Anonymized Auditor Prompt
# ==============================================================================

logger.info("Starting Engine 4: Applying Paper 2 Anonymized Constraint Routing...")

# We map the taxonomy exactly to our Pydantic Enum values to prevent manual mapping errors.
TAXONOMY_PROMPT = """
You are a deterministic code auditor operating under a strict Tree-of-Thoughts constraint.
You must evaluate the proposed code against its ORIGINAL SEMANTIC INTENT.
If the code violates the intent, classify it using ONLY the following exact string values:

- "Error_A_Input": missing or incorrect data type storage
- "Error_B_Output": non-compliant output format
- "Error_C_Variable": incorrect value stored
- "Error_D_Computation": incorrect calculation, wrong operation order, or wrong variable used in math
- "Error_E_Condition": incorrect if/else logic
- "Error_F_Branching": incorrect loop breaks
- "Error_G_Loop": incorrect iteration boundaries
- "Error_H_Array_String": incorrect indexing
- "Error_I_Function": incorrect parameter definitions
- "Error_J_Conceptual": breaks domain business logic or solves the wrong problem
- "No_Error": The code perfectly matches the semantic intent.
"""

# Production-safe Pydantic schema extraction (handles both V1 and V2)
try:
    json_schema = json.dumps(AuditResult.model_json_schema(), indent=2)
except AttributeError:
    json_schema = AuditResult.schema_json()

sys_prompt = f"{TAXONOMY_PROMPT}\n\nYou must output a valid JSON object matching this schema exactly:\n{json_schema}"

# Notice the carefully closed markdown backticks around {new_code}
user_prompt = f"""
ORIGINAL SEMANTIC INTENT:
\"{original_intent}\"

PROPOSED PULL REQUEST CODE:
```python
{new_code}
```
INSTRUCTIONS:
Simulate 3 expert developers analyzing the code step-by-step.
Compare the math and data flow in the PROPOSED CODE against the ORIGINAL SEMANTIC INTENT.
Determine if a logical error occurred based on the taxonomy.
Output the final JSON decision.
"""
audit_prompt = f"<|im_start|>system\n{sys_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n```json\n"

#3. Execute the Audit
logger.info("Executing Neuro-Symbolic Audit via vLLM on the A100...")
audit_params = SamplingParams(temperature=0.0, max_tokens=1024)
audit_outputs = llm.generate([audit_prompt], audit_params, use_tqdm=False)

#4. Strict Pydantic Validation
generated_text = audit_outputs[0].outputs[0].text
parsed_json = extract_json("```json\n" + generated_text)
print("\n" + "="*70)
print("🎯 NEURO-SYMBOLIC AUDIT RESULT 🎯")
print("="*70)
try:
    # We manually map the blast radius since it's a system-level variable, not AI-generated
    parsed_json['blast_radius_nodes'] = blast_radius

    # Run the raw JSON through our strict Pydantic model
    final_audit = AuditResult(**parsed_json)

    print(f"STATUS:      {final_audit.status}")
    print(f"ERROR TYPE:  {final_audit.error_type.name}")
    print(f"NODES:       {final_audit.blast_radius_nodes}")
    print(f"\nREASONING:\n{final_audit.reasoning}")

    if final_audit.status.upper() == "REJECTED":
        logger.warning(f"CI/CD PIPELINE BLOCKED: {final_audit.error_type.name} detected.")
    else:
        logger.success("CI/CD PIPELINE PASSED: Code safely verified.")
except Exception as e:
    logger.error(f"Pydantic Validation Failed. The LLM broke the schema constraint: {e}")
    print(f"Raw Output for debugging:\n{json.dumps(parsed_json, indent=2)}")
print("="*70 + "\n")

2026-05-07 22:42:03 | INFO     | __main__:<cell line: 0>:7 - Starting Engine 3: Extracting Graph Delta (Blast Radius)...
2026-05-07 22:42:03 | INFO     | __main__:<cell line: 0>:16 - Blast Radius identified: ['process_order', 'apply_discount', 'calculate_tax']
2026-05-07 22:42:03 | INFO     | __main__:<cell line: 0>:22 - Starting Engine 4: Applying Paper 2 Anonymized Constraint Routing...
2026-05-07 22:42:03 | INFO     | __main__:<cell line: 0>:69 - Executing Neuro-Symbolic Audit via vLLM on the A100...

🎯 NEURO-SYMBOLIC AUDIT RESULT 🎯
STATUS:      REJECTED
ERROR TYPE:  ERROR_J_CONCEPTUAL
NODES:       ['process_order', 'apply_discount', 'calculate_tax']

REASONING:
The proposed code does not include a function to calculate tax, which is a critical part of the original semantic intent. The function `calculate_tax` is referenced but not defined, and there is no indication of how the tax rate is applied. This breaks the domain business logic as it fails to add tax to the discounted price.